In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [0]:
"""Databricks Workflow entry point for the retail order quality lakehouse."""

from __future__ import annotations

from datetime import datetime, timezone

from order_quality.bronze import read_raw_orders
from order_quality.gold import daily_order_volume, revenue_by_region
from order_quality.quality import (
    apply_order_quality_checks,
    expected_order_schema,
    expected_raw_order_schema,
    passed_orders,
    quality_metrics_summary,
    quarantined_orders,
    schema_drift_checks,
)
from order_quality.silver import standardize_orders


dbutils.widgets.text("task", "all")
dbutils.widgets.text("raw_path", "/Volumes/workspace/default/retail_quality_raw")
dbutils.widgets.text("schema_name", "retail_quality")
dbutils.widgets.text("pipeline_run_id", "")

task = dbutils.widgets.get("task")
raw_path = dbutils.widgets.get("raw_path")
schema_name = dbutils.widgets.get("schema_name")
pipeline_run_id = dbutils.widgets.get("pipeline_run_id") or datetime.now(timezone.utc).strftime(
    "%Y%m%d%H%M%S"
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")


def write_table(df, table_name: str) -> None:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{schema_name}.{table_name}"
    )


def run_bronze() -> None:
    bronze_orders = read_raw_orders(spark, f"{raw_path}/orders/*.json")
    write_table(bronze_orders, "bronze_orders")


def run_silver() -> None:
    bronze_orders = spark.table(f"{schema_name}.bronze_orders")
    silver_orders = standardize_orders(bronze_orders)
    raw_schema_checks = schema_drift_checks(bronze_orders, expected_raw_order_schema())
    silver_schema_checks = schema_drift_checks(silver_orders, expected_order_schema())
    write_table(silver_orders, "silver_orders")
    write_table(raw_schema_checks.unionByName(silver_schema_checks), "schema_quality_checks")


def run_quality() -> None:
    silver_orders = spark.table(f"{schema_name}.silver_orders")
    schema_checks = spark.table(f"{schema_name}.schema_quality_checks")
    checked_orders = apply_order_quality_checks(silver_orders)
    clean_orders = passed_orders(checked_orders)
    quarantine = quarantined_orders(checked_orders, pipeline_run_id)
    metrics = quality_metrics_summary(checked_orders, quarantine, schema_checks, pipeline_run_id)
    write_table(clean_orders, "silver_orders_valid")
    write_table(quarantine, "quarantine_orders")
    write_table(metrics, "quality_metrics_summary")


def run_gold() -> None:
    valid_orders = spark.table(f"{schema_name}.silver_orders_valid")
    write_table(daily_order_volume(valid_orders), "gold_daily_order_volume")
    write_table(revenue_by_region(valid_orders), "gold_revenue_by_region")


if task in ("bronze", "all"):
    run_bronze()
if task in ("silver", "all"):
    run_silver()
if task in ("quality", "all"):
    run_quality()
if task in ("gold", "all"):
    run_gold()


In [0]:
display(spark.sql("SHOW TABLES IN retail_quality"))

database,tableName,isTemporary
retail_quality,bronze_orders,false
retail_quality,gold_daily_order_volume,false
retail_quality,gold_revenue_by_region,false
retail_quality,quality_metrics_summary,false
retail_quality,quarantine_orders,false
retail_quality,schema_quality_checks,false
retail_quality,silver_orders,false
retail_quality,silver_orders_valid,false
,_sqldf,true


In [0]:
display(spark.table("retail_quality.quarantine_orders"))

order_id,customer_id,order_ts,region,product_id,quantity,unit_price,discount_amount,status,_source_file,_ingestion_ts,is_duplicate_order_id,reason_codes,quality_status,order_amount,pipeline_run_id,quarantined_at
ORD-1006,null,2026-05-02T09:30:00.000Z,SOUTH,SKU-50,2,49.5,0.0,PAID,dbfs:/Volumes/workspace/default/retail_quality_raw/orders/day_02_bad_records.json,2026-05-29T20:07:55.915Z,false,List(NULL_CUSTOMER_ID),FAIL,99.0,20260529200752,2026-05-29T20:08:06.689Z
ORD-1007,C-007,null,WEST,SKU-60,1,35.0,0.0,SHIPPED,dbfs:/Volumes/workspace/default/retail_quality_raw/orders/day_02_bad_records.json,2026-05-29T20:07:55.915Z,false,List(NULL_OR_INVALID_ORDER_TS),FAIL,35.0,20260529200752,2026-05-29T20:08:06.689Z
ORD-1008,C-008,2026-05-02T13:00:00.000Z,MIDWEST,SKU-70,0,15.0,0.0,PAID,dbfs:/Volumes/workspace/default/retail_quality_raw/orders/day_02_bad_records.json,2026-05-29T20:07:55.915Z,false,List(INVALID_QUANTITY),FAIL,0.0,20260529200752,2026-05-29T20:08:06.689Z
ORD-1009,C-009,2026-05-02T13:30:00.000Z,SOUTH,SKU-20,1,199.99,0.0,UNKNOWN,dbfs:/Volumes/workspace/default/retail_quality_raw/orders/day_02_bad_records.json,2026-05-29T20:07:55.915Z,false,List(INVALID_STATUS),FAIL,199.99,20260529200752,2026-05-29T20:08:06.689Z
ORD-1010,C-010,2026-05-03T09:00:00.000Z,WEST,SKU-30,2,12.5,0.0,PAID,dbfs:/Volumes/workspace/default/retail_quality_raw/orders/day_03_duplicates_and_drift.json,2026-05-29T20:07:55.915Z,true,List(DUPLICATE_ORDER_ID),FAIL,25.0,20260529200752,2026-05-29T20:08:06.689Z
ORD-1010,C-010,2026-05-03T09:05:00.000Z,WEST,SKU-30,2,12.5,0.0,PAID,dbfs:/Volumes/workspace/default/retail_quality_raw/orders/day_03_duplicates_and_drift.json,2026-05-29T20:07:55.915Z,true,List(DUPLICATE_ORDER_ID),FAIL,25.0,20260529200752,2026-05-29T20:08:06.689Z


In [0]:
display(spark.table("retail_quality.quality_metrics_summary"))

pipeline_run_id,check_type,status,records_checked,failure_count,pass_rate,details,metric_date,created_at
20260529200752,INVALID_QUANTITY,FAIL,12,1,0.5,Failure count by reason code,2026-05-29,2026-05-29T20:08:08.653Z
20260529200752,NULL_CUSTOMER_ID,FAIL,12,1,0.5,Failure count by reason code,2026-05-29,2026-05-29T20:08:08.653Z
20260529200752,INVALID_STATUS,FAIL,12,1,0.5,Failure count by reason code,2026-05-29,2026-05-29T20:08:08.653Z
20260529200752,NULL_OR_INVALID_ORDER_TS,FAIL,12,1,0.5,Failure count by reason code,2026-05-29,2026-05-29T20:08:08.653Z
20260529200752,DUPLICATE_ORDER_ID,FAIL,12,2,0.5,Failure count by reason code,2026-05-29,2026-05-29T20:08:08.653Z
20260529200752,schema_extra_column,WARN,12,1,0.0,Schema contract checks,2026-05-29,2026-05-29T20:08:08.653Z
20260529200752,schema_type_match,PASS,12,18,1.0,Schema contract checks,2026-05-29,2026-05-29T20:08:08.653Z
20260529200752,row_quality,FAIL,12,6,0.5,Record-level gates before Gold,2026-05-29,2026-05-29T20:08:08.653Z


In [0]:
display(spark.table("retail_quality.gold_revenue_by_region"))

region,order_count,units_sold,gross_revenue
NORTHEAST,3,6,524.95
SOUTH,1,1,179.99
MIDWEST,1,1,80.0
WEST,1,4,50.0


In [0]:
display(spark.table("retail_quality.gold_daily_order_volume"))

order_date,order_count,gross_revenue,active_customers
2026-05-01,4,359.97,4
2026-05-02,1,74.97,1
2026-05-03,1,400.0,1
